# EICC GRPO Training Notebook

Colab-ready notebook for the Enterprise Incident Command Center:
1. Clone repo & install dependencies
2. Verify environment works (dry-run)
3. Run baseline evaluation
4. Train with GRPO
5. Run post-training evaluation
6. Plot reward curves and inspect artifacts

In [ ]:
# Step 1: Clone your repo
!git clone https://github.com/anujkamaljain/OpenEnv-Customer-Support.git
%cd OpenEnv-Customer-Support

In [ ]:
# Step 2: Install training dependencies
# If Colab asks for runtime restart, restart and rerun from Step 1.
# Pin pydantic/click for Colab compatibility and include GRPO transitive deps.
!pip install -q "pydantic>=2.12,<3" "click<8.2" unsloth "trl>=0.15" datasets peft bitsandbytes matplotlib mergekit llm-blender

In [ ]:
# Step 3: Verify environment works (quick sanity check)
!python train.py --iterations 1 --episodes 1 --k 2 --dry-run

In [ ]:
# Step 4: Run baseline evaluation (before training)
!python evaluate.py --policy baseline --episodes-per-difficulty 5 --output-dir artifacts/eval

In [ ]:
# Step 5: GRPO Training (~6-8 hours on T4, ~2-3 hours on V100)
# Reduce iterations/episodes if short on time:
#   --iterations 10 --episodes 15 --k 2  (~2-3 hours on T4)
!python train.py --iterations 20 --episodes 30 --k 4 --eval-episodes 5 --output-dir artifacts/train

In [ ]:
# Step 6: Run post-training comparison evaluation
!python evaluate.py --policy compare --episodes-per-difficulty 5 --plot --output-dir artifacts/eval

In [ ]:
# Step 7: View results
from pathlib import Path
import json

baseline_path = Path("artifacts/eval/baseline_report.json")
trained_path = Path("artifacts/eval/trained_report.json")
if baseline_path.exists() and trained_path.exists():
    baseline = json.loads(baseline_path.read_text(encoding="utf-8"))
    trained = json.loads(trained_path.read_text(encoding="utf-8"))
    print("Baseline avg_normalized_reward:", baseline["avg_normalized_reward"])
    print("Trained avg_normalized_reward:", trained["avg_normalized_reward"])
    print("Behavior examples:")
    for line in trained.get("behavior_examples", []):
        print("-", line)
else:
    print("Run evaluation cells first.")

In [ ]:
# Step 8: Display reward curve
from pathlib import Path
from IPython.display import Image, display

curve = Path("artifacts/eval/reward_curves.png")
if curve.exists():
    display(Image(filename=str(curve)))
else:
    print("Run compare evaluation with --plot first.")